# The SMA Crossover, Honestly Backtested: QQQ and Bitcoin, Costs Included

Buy when the 50-day average crosses above the 200-day; step aside when it crosses back. That is the **golden cross** — the systematic trend-following workhorse: one rule, two parameters, a century of folklore. Its simplicity is exactly why it makes the perfect classroom for backtesting discipline 101 — every sin that quietly ruins real backtests fits on one screen of code here, where you can watch it happen. We run the same long-or-flat rule on QQQ and BTC-USD over 2015–2024 with next-day execution and 10bp per-side costs, then do what most crossover backtests skip: sweep the parameter grid, stress the costs, and admit what one in-sample decade can and cannot prove.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import vectorbt as vbt

plt.rcParams["figure.figsize"] = (10, 5)

## 1. The rule: two moving averages

Compute a fast and a slow simple moving average of the close. Think of the slow one as the market's long memory and the fast one as its recent mood. When the fast SMA sits above the slow one, recent prices sit above the long trend — be long. When it drops below, be flat. No shorting, no leverage, no discretion, no opinions. With `fast = 50` and `slow = 200` this is the golden cross of financial television fame — and, more respectably, a one-asset special case of **time-series momentum**, the effect Moskowitz, Ooi and Pedersen documented across 58 futures markets.

We run it on two deliberately different animals: the Nasdaq-100 ETF (exchange calendar, ~252 bars a year) and Bitcoin (trades every day of the week, ~365 bars a year). BTC has prices on weekends where QQQ has none, so each asset is backtested on its own calendar — per-asset `dropna`, per-asset annualisation — never forward-filled onto a shared grid.

In [ ]:
raw = yf.download(["QQQ", "BTC-USD"], start="2015-01-01", end="2025-01-01",
                  auto_adjust=True, progress=False)["Close"]

qqq = raw["QQQ"].dropna()      # ~252 bars / year
btc = raw["BTC-USD"].dropna()  # ~365 bars / year
print(len(qqq), "QQQ bars ·", len(btc), "BTC bars")

## 2. Next-day execution — the shift(1) that keeps you honest

The most important line in the backtest is not the signal. It is the shift. Today's SMA cross is computed on today's **close**— a price you cannot trade before you have seen it. So today's signal may only earn **tomorrow's** return. Drop the shift and the backtest buys every up-day one bar early: a lookahead bias that flatters almost any signal and quietly fabricates Sharpe.

Two more discipline details. Every position change pays 10 bp — entries and exits both. And the first 250 bars of each asset are reserved for SMA warm-up, so every parameter pair we test later is judged on the **same** evaluation window: 2015-12-30 → 2024-12-31 for QQQ, 2015-09-08 → 2024-12-31 for BTC. The pandas core above is deliberately transparent; we rebuild the identical trades with vectorbt's `Portfolio.from_signals` as an independent referee — final value, max drawdown and trade count agree.

In [ ]:
FAST, SLOW, FEE, WARMUP = 50, 200, 0.001, 250

def backtest(px, fast=FAST, slow=SLOW, fee=FEE):
    signal   = (px.rolling(fast).mean() > px.rolling(slow).mean()).astype(float)
    position = signal.shift(1).fillna(0.0)   # <- no lookahead. The whole game.
    ret      = px.pct_change().fillna(0.0)
    cost     = position.diff().abs().fillna(0.0) * fee
    strat    = position * ret - cost
    return strat.iloc[WARMUP:], position.iloc[WARMUP:]

def stats(rets, ann):
    eq = (1 + rets).cumprod()
    sh = rets.mean() / rets.std(ddof=1) * np.sqrt(ann)
    return {"annRet": eq.iloc[-1] ** (ann / len(rets)) - 1,
            "annVol": rets.std(ddof=1) * np.sqrt(ann),
            "sharpe": sh,
            "maxDD": (eq / eq.cummax() - 1).min()}

## 3. Same rule, two verdicts

Growth of $100, log scale (BTC would render every other line invisible otherwise). Watch **where** the strategy detaches from buy-and-hold: always in the big drawdowns. A slow trend filter is not an accelerator — it is a brake pedal.

- **QQQ** — lost the return race ( $439 vs $478 ), but at lower vol and a third-shallower worst drawdown, for a slightly better Sharpe. No return edge; a brake pedal.
- **BTC** — kept pace with one of the great bull markets in modern data while sitting out 36% of all days, and cut the max drawdown from -83.4% to -69.3% .

Neither run is a money machine; both are drawdown insurance. Two conservatisms are baked into every number above: flat days are credited **nothing**(park the cash at the T-bill rate and the strategy line improves while buy-and-hold's cannot), and Sharpe is quoted on raw rather than excess returns. Both choices shade against the strategy, which is the right direction to be wrong in.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (name, px, ann) in zip(axes, [("QQQ", qqq, 252), ("BTC-USD", btc, 365)]):
    strat, pos = backtest(px)
    bh = px.pct_change().fillna(0.0).iloc[WARMUP:]
    (100 * (1 + strat).cumprod()).plot(ax=ax, label="50/200 crossover")
    (100 * (1 + bh).cumprod()).plot(ax=ax, label="buy & hold", alpha=0.7)
    ax.set_yscale("log"); ax.set_title(name); ax.legend()

    s, b = stats(strat, ann), stats(bh, ann)
    n_tr = int((pos.diff() > 0).sum() + (pos.iloc[0] > 0))
    print(f"{name:8s} strat: Sharpe {s['sharpe']:.2f} maxDD {s['maxDD']:+.1%} "
          f"({n_tr} trades, {pos.mean():.0%} in market)  |  "
          f"b&h: Sharpe {b['sharpe']:.2f} maxDD {b['maxDD']:+.1%}")
plt.show()

In [ ]:
for name, px in [("QQQ", qqq), ("BTC-USD", btc)]:
    sig  = px.rolling(FAST).mean() > px.rolling(SLOW).mean()
    prev = sig.shift(1).fillna(False)
    entries = (sig & ~prev).iloc[WARMUP:].copy()
    exits   = (~sig & prev).iloc[WARMUP:].copy()
    if bool(sig.iloc[WARMUP - 1]):
        entries.iloc[0] = True   # carry the already-open position into the window
    pf = vbt.Portfolio.from_signals(px.iloc[WARMUP:], entries, exits,
                                    fees=FEE, init_cash=100, freq="1D")
    print(f"{name:8s} vectorbt: final ${pf.final_value():,.0f}  "
          f"maxDD {pf.max_drawdown():+.1%}  trades {pf.trades.count()}")

In [ ]:
from pyfolio import timeseries

strat_q, _ = backtest(qqq)
timeseries.perf_stats(strat_q).loc[["Annual return", "Annual volatility",
                                    "Sharpe ratio", "Max drawdown"]]

## 4. Discipline 101: the parameter grid

One backtest is an anecdote. If 50/200 were genuinely special, small changes to the parameters should not change the story — real effects are smooth neighbourhoods, and only overfit ones are sharp peaks. So before believing 50/200, ask whether the **neighbourhood** agrees: we sweep fast ∈ { 10, 20, 50, 100 } × slow ∈ { 100, 150, 200, 250 } — 15 valid systems per asset, identical execution, identical costs — and heat-map the Sharpe. Both grids share one colour scale, so a glance tells you which asset rewarded trend.

This is the card's thesis in one picture. **QQQ**: every cell lands between 0.78 and 1.01 , straddling buy-and-hold's 0.90 — only 7 of 15 cells beat it, none decisively. Picking the best cell after the fact and calling it edge is selection bias, not alpha; on this asset the honest claim is "no return edge, a consistent drawdown brake". **BTC**: 12 of 15 cells sit above buy-and-hold's 1.27 , the whole grid stays in the 1.18 – 1.43 band, and every cell slashes the max drawdown. The same rule gets two different verdicts because **trend behaves differently per asset**— Bitcoin's decade delivered longer, cleaner trends and far deeper crashes for the filter to sidestep.

In [ ]:
FASTS, SLOWS = [10, 20, 50, 100], [100, 150, 200, 250]

def sharpe_grid(px, ann):
    g = np.full((len(FASTS), len(SLOWS)), np.nan)
    for i, f in enumerate(FASTS):
        for j, s in enumerate(SLOWS):
            if f >= s:
                continue
            rets, _ = backtest(px, f, s)
            g[i, j] = rets.mean() / rets.std(ddof=1) * np.sqrt(ann)
    return g

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (name, px, ann) in zip(axes, [("QQQ", qqq, 252), ("BTC-USD", btc, 365)]):
    g = sharpe_grid(px, ann)
    bh = px.pct_change().fillna(0.0).iloc[WARMUP:]
    bh_sh = bh.mean() / bh.std(ddof=1) * np.sqrt(ann)
    im = ax.imshow(g, cmap="RdYlGn", vmin=0.6, vmax=1.5)
    for i in range(len(FASTS)):
        for j in range(len(SLOWS)):
            if not np.isnan(g[i, j]):
                ax.text(j, i, f"{g[i, j]:.2f}", ha="center", va="center", fontsize=9)
    ax.set_xticks(range(len(SLOWS)), SLOWS); ax.set_yticks(range(len(FASTS)), FASTS)
    ax.set_xlabel("slow"); ax.set_ylabel("fast")
    ax.set_title(f"{name} — Sharpe (buy-hold = {bh_sh:.2f})")
plt.colorbar(im, ax=axes, shrink=0.8);
plt.show()

## 5. The costs dial: 0 / 10 / 25 bp

Costs are the second-biggest backtest killer after lookahead — but they bite in proportion to turnover. The 50/200 pair trades a handful of times a decade, so even 25bp per side barely dents it. Speed the system up to 10/100 and the toll booth opens:

Slow trend is nearly free to run. Fast trend pays a visible tax — QQQ's 10/100 variant loses 0.06 Sharpe going from free execution to 25bp, on 18 round-trip entries. Any strategy whose backtest only works at 0bp does not work.

In [ ]:
rows = []
for name, px, ann in [("QQQ", qqq, 252), ("BTC-USD", btc, 365)]:
    for f, s in [(50, 200), (10, 100)]:
        row = {"asset": name, "pair": f"{f}/{s}"}
        for bp in (0, 10, 25):
            rets, pos = backtest(px, f, s, fee=bp / 1e4)
            row[f"Sharpe@{bp}bp"] = round(rets.mean() / rets.std(ddof=1) * np.sqrt(ann), 2)
        row["trades"] = int((pos.diff() > 0).sum() + (pos.iloc[0] > 0))
        rows.append(row)
pd.DataFrame(rows).set_index(["asset", "pair"])

## 6. One sample, no walk-forward

Everything above is one ten-year window, evaluated in-sample. There is no walk-forward, no out-of-sample holdout, and no multiple-testing haircut for the 15 variants we just eyeballed per asset. 2015–2024 handed both assets two of the strongest trend decades they have ever printed — a regime gift the next decade owes nobody. Before promoting any cell of that grid, run the standard honesty checklist: point-in-time data, walk-forward splits, pessimistic costs, and a **deflated Sharpe**— Bailey & López de Prado's correction for exactly the selection bias a 15 -cell grid search manufactures — on whatever looked best.

- Brock, W., Lakonishok, J. & LeBaron, B. (1992). Simple Technical Trading Rules and the Stochastic Properties of Stock Returns. Journal of Finance 47(5).
- Moskowitz, T., Ooi, Y.H. & Pedersen, L.H. (2012). Time Series Momentum. Journal of Financial Economics 104(2).
- Bailey, D.H. & López de Prado, M. (2014). The Deflated Sharpe Ratio: Correcting for Selection Bias, Backtest Overfitting and Non-Normality. Journal of Portfolio Management 40(5).
- vectorbt documentation — `Portfolio.from_signals`, the signal-based backtesting API used as the fill cross-check.
- Companion notebook: `sma-crossover-backtest.ipynb` — reproduces every figure from raw data (deterministic, no simulation).